### ⚠️ Patched for Local Execution
This notebook was originally designed for Google Colab. It has been automatically patched:
- Google Colab-specific imports and `drive.mount()` calls have been commented out
- Colab file paths (`/content/drive/...`) have been replaced with relative paths (`./`)
- `!pip install` commands have been commented out (install packages in your venv instead)

**To run locally:** activate your Python virtual environment first, then run this notebook in VS Code or Jupyter.

In [ ]:
# [PATCHED] from google.colab import drive
# [PATCHED] drive.mount('./')

In [ ]:
import os
from keras.preprocessing import text_dataset_from_directory
import tensorflow as tf

In [ ]:
data_path = './ Notebooks/Hsoub/L10'
labels = os.listdir(data_path)
batch_size =64
raw_data = text_dataset_from_directory(
    directory=data_path,
    batch_size= batch_size,
)

In [ ]:
print("Article classes are:\n",raw_data.class_names)
num_classes= len(raw_data.class_names)

In [ ]:
X=[]
y_labels=[]
y=[]
for text_batch, label_batch in raw_data:

    text_batch=text_batch.numpy()
    label_batch=label_batch.numpy()

    for i in range(len(text_batch)):

        text=text_batch[i].decode("utf-8")

        X.append(text)

        c_index=label_batch[i]
        y.append(c_index)
        class_label = raw_data.class_names[c_index]
        y_labels.append(class_label)
print(len(X))
print(len(y))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

unique, counts = np.unique(y_labels, return_counts=True)
plt.figure("classe Pie", figsize=(7, 7))
plt.title("Pie plot of the class frequencies")
plt.pie(counts, labels=unique)

plt.show()

In [ ]:
import nltk
nltk.download('stopwords')
nltk.download('punkt')
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import string

listStopwords = stopwords.words('arabic')
def cleanText(text):
    numbers="0123456789"
    arabic_punctuation='''`÷«»×؛<>_()*^ـ،/:"؟.,'~¦+|!”…“–ـ'''
    english_punctuation=string.punctuation
    del_chars=english_punctuation+arabic_punctuation+numbers
    for char in del_chars:
        text = text.replace(char, " ")
    text = text.strip(' ')
    tokens_list=word_tokenize(text)
    filtered = []
    for txt in tokens_list:
        if txt not in listStopwords:
            filtered.append(txt)
    text = ' '.join(filtered)
    return text

In [ ]:
import random
import numpy as np
import tensorflow as tf
seed_value = 42
np.random.seed(seed_value)
tf.random.set_seed(seed_value)
random.seed(seed_value)

In [ ]:
import random
samples=10000

random_indices = random.sample(range(len(X)), samples)

X = [X[i] for i in random_indices]
y = [y[i] for i in random_indices]

In [ ]:
X[1]

In [ ]:
X = [cleanText(text) for text in X]
X[1]

In [ ]:
#https://huggingface.co/asafaya

In [ ]:
#https://sparknlp.org/2022/04/11/bert_embeddings_bert_base_arabic_ar_3_0.html

In [ ]:
#https://huggingface.co/asafaya/bert-base-arabic

In [ ]:
bert_model_name="asafaya/bert-base-arabic"

In [ ]:
from transformers import BertTokenizer

tokenizer = BertTokenizer.from_pretrained(bert_model_name)


In [ ]:
from transformers import TFBertModel
bert_model = TFBertModel.from_pretrained(bert_model_name)

In [ ]:
X_bert_dict = tokenizer(X, return_tensors="tf", padding=True, truncation=True, max_length = 128)
X_bert_dict

In [ ]:
tokens_ids=X_bert_dict["input_ids"].numpy()
tokens_ids

In [ ]:
X_embeddings_list = []
for i in range(0, len(tokens_ids), batch_size):

    batch_segments = tokens_ids[i:i+batch_size]

    batch_segments_tf = tf.convert_to_tensor(batch_segments)

    dict={"input_ids": batch_segments_tf}

    embeddings = bert_model(dict)["last_hidden_state"]

    X_embeddings_list.append(embeddings)

In [ ]:

X_embeddings = tf.concat(X_embeddings_list, axis=0)

print("Shape of concatenated embeddings:", X_embeddings.shape)

In [ ]:
X_embeddings_np = X_embeddings.numpy()
y_np =np.array(y)

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_embeddings_np, y_np, random_state=42)

In [ ]:
from keras.layers import Input, Flatten, Dense,Dropout
from keras.models import Sequential
import tensorflow as tf

model = Sequential()

model.add(Input(shape=(X_embeddings_np.shape[1], X_embeddings_np.shape[2])))

model.add(Flatten())
model.add(Dropout(0.5))
model.add(Dense(num_classes, activation='softmax'))

model.compile(optimizer="adam", loss='sparse_categorical_crossentropy', metrics=['accuracy'])

epochs = 20

history = model.fit(X_train, y_train, epochs=epochs, batch_size=batch_size,
                    validation_data=(X_test, y_test))


In [ ]:
train_accuracy = history.history['accuracy']
val_accuracy = history.history['val_accuracy']
print("Train Accuracy:", round(100*train_accuracy[-1],2))
print("Validation Accuracy:", round(100*val_accuracy[-1],2))


In [ ]:
plt.plot(history.history['accuracy'])
plt.plot(history.history['val_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Train', 'Val'], loc='lower right')
plt.show()

In [ ]:
y_pred = model.predict(X_test)
y_pred

In [ ]:
y_pred = np.argmax(y_pred, axis=1)
y_pred

In [ ]:
from sklearn.metrics import confusion_matrix
cm = confusion_matrix(y_test, y_pred)
cm


In [ ]:
from sklearn.metrics import  ConfusionMatrixDisplay
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=raw_data.class_names)
disp.plot(cmap=plt.cm.Blues, values_format='d')
plt.title('Confusion Matrix')
plt.show()
